In [0]:
def display_files_distribution_for_table(table: str):
    display(spark.sql(f'''
            SELECT _metadata.file_path AS file, count(*) AS rows
            FROM {table}
            GROUP BY _metadata.file_path
            ORDER BY file'''))

Let's generate some random but unbalanced data first. 

In [0]:
tables_to_drop = ['events_source_coalesced', 'events_source_rebalanced', 'events_source_rebalanced_country', 'events_source_repartitioned_2', 'events_source_repartitioned_country', 'events_source_repartitioned_country_4', 'events_source_repartitioned_range', 'events_source']

for table in tables_to_drop:
    spark.sql(f'DROP TABLE IF EXISTS {table}')


In [0]:
%sql
CREATE TABLE events_source AS
SELECT
  id,
  CASE
    WHEN rand(42) < 0.80 THEN 'US'
    WHEN rand(42) < 0.85 THEN 'DE'
    WHEN rand(42) < 0.90 THEN 'PL'
    WHEN rand(42) < 0.95 THEN 'JP'
    ELSE 'BR'
  END                                     AS country,
  CAST(rand(7) * 10000 AS DECIMAL(10,2))  AS amount,
  current_timestamp() - make_interval(0, 0, 0, CAST(rand(1) * 365 AS INT), 0, 0, 0) AS event_ts
FROM range(0, 1000);


num_affected_rows,num_inserted_rows


You should see the US having much more rows than other countries

In [0]:
%sql
SELECT country, count(*) AS cnt
FROM events_source
GROUP BY country
ORDER BY cnt DESC;

country,cnt
US,799
DE,173
PL,25
JP,2
BR,1


...but there is no difference at the storage level. The table created 8 files, each storing 125 rows.

In [0]:
%sql
SELECT _metadata.file_path AS file, count(*) AS rows
FROM events_source
GROUP BY _metadata.file_path
ORDER BY file;


file,rows
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/517eeec8-77b6-4174-ab67-4a65a7647c7b/part-00000-94debff3-26f2-44ff-8570-7a30581b8545.c000.zstd.parquet,125
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/517eeec8-77b6-4174-ab67-4a65a7647c7b/part-00001-123442c8-0422-43e2-a683-b6ab724ff366.c000.zstd.parquet,125
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/517eeec8-77b6-4174-ab67-4a65a7647c7b/part-00002-0e915de8-2595-4e61-9a76-b7af10549f83.c000.zstd.parquet,125
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/517eeec8-77b6-4174-ab67-4a65a7647c7b/part-00003-70cf542c-89c6-4f02-9e5c-30bd39a305b8.c000.zstd.parquet,125
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/517eeec8-77b6-4174-ab67-4a65a7647c7b/part-00004-09013033-9f65-47bd-9e72-b92953db2922.c000.zstd.parquet,125
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/517eeec8-77b6-4174-ab67-4a65a7647c7b/part-00005-82059355-7cd3-4ceb-bd74-71b46c1df8ec.c000.zstd.parquet,125
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/517eeec8-77b6-4174-ab67-4a65a7647c7b/part-00006-8ccef12a-3fa6-4078-a248-d5e6029b0a84.c000.zstd.parquet,125
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/517eeec8-77b6-4174-ab67-4a65a7647c7b/part-00007-59d96110-fff8-48cd-a8c3-ee46bfb82acb.c000.zstd.parquet,125


# Partitioning hints
An easy way to see how partitioning hints impact query execution is tables creation. In the cells below you'll see how applying different partitioning hints modify data distribution of the newly created tables. 

## Coalesce hint
The coalece hint will reduce the number of partitions. An expected output on the created table is less than 8 files that you can see in the original table.

In [0]:
%sql
EXPLAIN FORMATTED
SELECT /*+ COALESCE(2) */ *
FROM events_source;

plan
"== Physical Plan == Coalesce (4) +- * ColumnarToRow (3) +- PhotonResultStage (2) +- PhotonScan parquet workspace.default.events_source (1) (1) PhotonScan parquet workspace.default.events_source Output [4]: [id#13678L, country#13679, amount#13680, event_ts#13681] Location: PreparedDeltaFileIndex [s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/517eeec8-77b6-4174-ab67-4a65a7647c7b] ReadSchema: struct (2) PhotonResultStage Input [4]: [id#13678L, country#13679, amount#13680, event_ts#13681] (3) ColumnarToRow [codegen id : 1] Input [4]: [id#13678L, country#13679, amount#13680, event_ts#13681] (4) Coalesce Input [4]: [id#13678L, country#13679, amount#13680, event_ts#13681] Arguments: 2 == Photon Explanation == Photon does not fully support the query because: Unsupported node: Coalesce 2. Reference node: Coalesce 2 == Optimizer Statistics (table names per statistics state) == missing = partial = full = events_source"


In [0]:
%sql
CREATE OR REPLACE TABLE events_source_coalesced AS
SELECT /*+ COALESCE(2) */ * FROM events_source;

num_affected_rows,num_inserted_rows


In [0]:
display_files_distribution_for_table('events_source_coalesced')

file,rows
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/09675869-e56d-4326-831a-2fedb35e1f96/part-00000-e08fb0bd-d46f-4c14-8af0-14a7020e5680.c000.zstd.parquet,1000


## Repartition hint
The repartition hint reshuffles data. Consequently, the number of files should correspond to the number of shuffle partitions.

In [0]:
%sql
EXPLAIN FORMATTED
SELECT /*+ REPARTITION(2) */ *
FROM events_source;


plan
"== Physical Plan == AdaptiveSparkPlan (8) +- == Initial Plan == PhotonResultStage (7) +- PhotonColumnarToRow (6) +- PhotonShuffleExchangeSource (5) +- PhotonShuffleMapStage (4) +- PhotonShuffleExchangeSink (3) +- PhotonSort (2) +- PhotonScan parquet workspace.default.events_source (1) (1) PhotonScan parquet workspace.default.events_source Output [4]: [id#14032L, country#14033, amount#14034, event_ts#14035] Location: PreparedDeltaFileIndex [s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/517eeec8-77b6-4174-ab67-4a65a7647c7b] ReadSchema: struct (2) PhotonSort Input [4]: [id#14032L, country#14033, amount#14034, event_ts#14035] Arguments: [id#14032L ASC NULLS FIRST, country#14033 ASC NULLS FIRST, amount#14034 ASC NULLS FIRST, event_ts#14035 ASC NULLS FIRST] (3) PhotonShuffleExchangeSink Input [4]: [id#14032L, country#14033, amount#14034, event_ts#14035] Arguments: RoundRobinPartitioning(2) (4) PhotonShuffleMapStage Input [4]: [id#14032L, country#14033, amount#14034, event_ts#14035] Arguments: REPARTITION_BY_NUM, [id=#8864] (5) PhotonShuffleExchangeSource Input [4]: [id#14032L, country#14033, amount#14034, event_ts#14035] (6) PhotonColumnarToRow Input [4]: [id#14032L, country#14033, amount#14034, event_ts#14035] (7) PhotonResultStage Input [4]: [id#14032L, country#14033, amount#14034, event_ts#14035] (8) AdaptiveSparkPlan Output [4]: [id#14032L, country#14033, amount#14034, event_ts#14035] Arguments: isFinalPlan=false == Photon Explanation == The query is fully supported by Photon. == Optimizer Statistics (table names per statistics state) == missing = partial = full = events_source"


In [0]:
%sql
CREATE OR REPLACE TABLE events_source_repartitioned_2 AS
SELECT /*+ REPARTITION(2) */ * FROM events_source;

num_affected_rows,num_inserted_rows


In [0]:
display_files_distribution_for_table('events_source_repartitioned_2')

file,rows
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/70f49325-1674-4ccf-b6d0-89a61daed120/part-00000-f75fa049-8ddc-4b88-9104-c2e75f1788ad.c000.zstd.parquet,501
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/70f49325-1674-4ccf-b6d0-89a61daed120/part-00001-27b67ceb-3bb7-4984-a402-66a536460833.c000.zstd.parquet,499


## Repartition hint by number and column
The repartition hint can also be modified to include a column. In this example, the created table will have 4 partitions, each storing a separate country*.


\* - in theory, in practice AQE partition coalesce will combine small partition with _JP_ data with a bigger partition storing _DE_ data

In [0]:
%sql
EXPLAIN FORMATTED
SELECT /*+ REPARTITION(4, country) */ *
FROM events_source;


plan
"== Physical Plan == AdaptiveSparkPlan (7) +- == Initial Plan == PhotonResultStage (6) +- PhotonColumnarToRow (5) +- PhotonShuffleExchangeSource (4) +- PhotonShuffleMapStage (3) +- PhotonShuffleExchangeSink (2) +- PhotonScan parquet workspace.default.events_source (1) (1) PhotonScan parquet workspace.default.events_source Output [4]: [id#11360L, country#11361, amount#11362, event_ts#11363] Location: PreparedDeltaFileIndex [s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/0531eab6-ec22-4de1-9f3e-58bf53f0079c] ReadSchema: struct (2) PhotonShuffleExchangeSink Input [4]: [id#11360L, country#11361, amount#11362, event_ts#11363] Arguments: hashpartitioning(country#11361, 4) (3) PhotonShuffleMapStage Input [4]: [id#11360L, country#11361, amount#11362, event_ts#11363] Arguments: REPARTITION_BY_NUM, [id=#7229] (4) PhotonShuffleExchangeSource Input [4]: [id#11360L, country#11361, amount#11362, event_ts#11363] (5) PhotonColumnarToRow Input [4]: [id#11360L, country#11361, amount#11362, event_ts#11363] (6) PhotonResultStage Input [4]: [id#11360L, country#11361, amount#11362, event_ts#11363] (7) AdaptiveSparkPlan Output [4]: [id#11360L, country#11361, amount#11362, event_ts#11363] Arguments: isFinalPlan=false == Photon Explanation == The query is fully supported by Photon. == Optimizer Statistics (table names per statistics state) == missing = partial = full = events_source"


In [0]:
%sql
CREATE OR REPLACE TABLE events_source_repartitioned_country_4 AS
SELECT /*+ REPARTITION(4, country) */ * FROM events_source;

num_affected_rows,num_inserted_rows


In [0]:
display_files_distribution_for_table('events_source_repartitioned_country_4')
"""
Initial data distribution; JP is stored alongside DE in the repartitioned table:
country	cnt
US	    799
DE	    173
PL	    25
JP	    2
BR	    1
"""

file,rows
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/993079ca-9d72-43ec-8ffe-798bce8cbf67/part-00000-e2add464-d8ec-4884-9a6a-b1c85b3c26c0.c000.zstd.parquet,799
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/993079ca-9d72-43ec-8ffe-798bce8cbf67/part-00001-80f5e50e-38a7-4c16-b6b8-a29f84f0c050.c000.zstd.parquet,1
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/993079ca-9d72-43ec-8ffe-798bce8cbf67/part-00002-7a1b363b-18a7-4009-b35d-661a7edb8c85.c000.zstd.parquet,25
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/993079ca-9d72-43ec-8ffe-798bce8cbf67/part-00003-45faff98-901c-426b-a657-eb1cb4aa795f.c000.zstd.parquet,175


'\nInitial data distribution; JP is stored alongside DE in the repartitioned table:\ncountry\tcnt\nUS\t    799\nDE\t    173\nPL\t    25\nJP\t    2\nBR\t    1\n'

## Repartition by range hint
Here we're going to create 4 files based on the range of the _amount_ column. In the output you should see the not overlapping ranges in each of the created files.

In [0]:
%sql
EXPLAIN FORMATTED
SELECT /*+ REPARTITION_BY_RANGE(4, amount) */ *
FROM events_source;


plan
"== Physical Plan == AdaptiveSparkPlan (7) +- == Initial Plan == PhotonResultStage (6) +- PhotonColumnarToRow (5) +- PhotonShuffleExchangeSource (4) +- PhotonShuffleMapStage (3) +- PhotonShuffleExchangeSink (2) +- PhotonScan parquet workspace.default.events_source (1) (1) PhotonScan parquet workspace.default.events_source Output [4]: [id#14718L, country#14719, amount#14720, event_ts#14721] Location: PreparedDeltaFileIndex [s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/517eeec8-77b6-4174-ab67-4a65a7647c7b] ReadSchema: struct (2) PhotonShuffleExchangeSink Input [4]: [id#14718L, country#14719, amount#14720, event_ts#14721] Arguments: rangepartitioning(amount#14720 ASC NULLS FIRST, 4) (3) PhotonShuffleMapStage Input [4]: [id#14718L, country#14719, amount#14720, event_ts#14721] Arguments: REPARTITION_BY_NUM, [id=#9429] (4) PhotonShuffleExchangeSource Input [4]: [id#14718L, country#14719, amount#14720, event_ts#14721] (5) PhotonColumnarToRow Input [4]: [id#14718L, country#14719, amount#14720, event_ts#14721] (6) PhotonResultStage Input [4]: [id#14718L, country#14719, amount#14720, event_ts#14721] (7) AdaptiveSparkPlan Output [4]: [id#14718L, country#14719, amount#14720, event_ts#14721] Arguments: isFinalPlan=false == Photon Explanation == The query is fully supported by Photon. == Optimizer Statistics (table names per statistics state) == missing = partial = full = events_source"


In [0]:
%sql
CREATE OR REPLACE TABLE events_source_repartitioned_range AS
SELECT /*+ REPARTITION_BY_RANGE(4, amount) */ * FROM events_source;

num_affected_rows,num_inserted_rows


In [0]:
display_files_distribution_for_table('events_source_repartitioned_range')

file,rows
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/23b44b0a-e3d3-4733-8607-879167958aaf/part-00000-6161ff58-6b63-40dd-af4c-280e44538ee5.c000.zstd.parquet,250
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/23b44b0a-e3d3-4733-8607-879167958aaf/part-00001-58c7b6e2-cca6-40a7-a97d-e1bd993faa9c.c000.zstd.parquet,250
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/23b44b0a-e3d3-4733-8607-879167958aaf/part-00002-754c90c6-96bd-421e-aedb-3b922334a254.c000.zstd.parquet,250
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/23b44b0a-e3d3-4733-8607-879167958aaf/part-00003-4e2232ec-2b66-4014-a8ab-2ebcdf9a3974.c000.zstd.parquet,250


In [0]:
%sql
SELECT _metadata.file_path AS file, count(*) AS rows, MIN(amount), MAX(amount)
FROM events_source_repartitioned_range
GROUP BY _metadata.file_path
ORDER BY file

file,rows,MIN(amount),MAX(amount)
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/23b44b0a-e3d3-4733-8607-879167958aaf/part-00000-6161ff58-6b63-40dd-af4c-280e44538ee5.c000.zstd.parquet,250,7.66,2232.08
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/23b44b0a-e3d3-4733-8607-879167958aaf/part-00001-58c7b6e2-cca6-40a7-a97d-e1bd993faa9c.c000.zstd.parquet,250,2245.02,4562.19
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/23b44b0a-e3d3-4733-8607-879167958aaf/part-00002-754c90c6-96bd-421e-aedb-3b922334a254.c000.zstd.parquet,250,4566.06,7167.43
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/23b44b0a-e3d3-4733-8607-879167958aaf/part-00003-4e2232ec-2b66-4014-a8ab-2ebcdf9a3974.c000.zstd.parquet,250,7209.26,9963.54


## Rebalance hint
The rebalance hint is the suggestion to apply the most optimal writing strategy. Since our example is relatively small, the hint will lead to creating a single file in the created table.

In [0]:
%sql
EXPLAIN FORMATTED
SELECT /*+ REBALANCE */ *
FROM events_source;


plan
"== Physical Plan == AdaptiveSparkPlan (8) +- == Initial Plan == PhotonResultStage (7) +- PhotonColumnarToRow (6) +- PhotonShuffleExchangeSource (5) +- PhotonShuffleMapStage (4) +- PhotonShuffleExchangeSink (3) +- PhotonSort (2) +- PhotonScan parquet workspace.default.events_source (1) (1) PhotonScan parquet workspace.default.events_source Output [4]: [id#15170L, country#15171, amount#15172, event_ts#15173] Location: PreparedDeltaFileIndex [s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/517eeec8-77b6-4174-ab67-4a65a7647c7b] ReadSchema: struct (2) PhotonSort Input [4]: [id#15170L, country#15171, amount#15172, event_ts#15173] Arguments: [id#15170L ASC NULLS FIRST, country#15171 ASC NULLS FIRST, amount#15172 ASC NULLS FIRST, event_ts#15173 ASC NULLS FIRST] (3) PhotonShuffleExchangeSink Input [4]: [id#15170L, country#15171, amount#15172, event_ts#15173] Arguments: RoundRobinPartitioning(1024) (4) PhotonShuffleMapStage Input [4]: [id#15170L, country#15171, amount#15172, event_ts#15173] Arguments: REBALANCE_PARTITIONS_BY_NONE, [id=#9852] (5) PhotonShuffleExchangeSource Input [4]: [id#15170L, country#15171, amount#15172, event_ts#15173] (6) PhotonColumnarToRow Input [4]: [id#15170L, country#15171, amount#15172, event_ts#15173] (7) PhotonResultStage Input [4]: [id#15170L, country#15171, amount#15172, event_ts#15173] (8) AdaptiveSparkPlan Output [4]: [id#15170L, country#15171, amount#15172, event_ts#15173] Arguments: isFinalPlan=false == Photon Explanation == The query is fully supported by Photon. == Optimizer Statistics (table names per statistics state) == missing = partial = full = events_source"


In [0]:
%sql
CREATE OR REPLACE TABLE events_source_rebalanced AS
SELECT /*+ REBALANCE */ * FROM events_source;

num_affected_rows,num_inserted_rows


In [0]:
display_files_distribution_for_table('events_source_rebalanced')

file,rows
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/4528a612-6443-41f1-a2ee-4a2390211fa7/part-00000-9c0fa7d8-099d-4bf2-bb61-3d06d08e59b6.c000.zstd.parquet,1000


## Rebalance hint with column
Even though we declare a column in the rebalance hint, it will still pick the most optimized strategy which in our small data case is a single file.

In [0]:
%sql
EXPLAIN FORMATTED
SELECT /*+ REBALANCE(country) */ *
FROM events_source;


plan
"== Physical Plan == AdaptiveSparkPlan (7) +- == Initial Plan == PhotonResultStage (6) +- PhotonColumnarToRow (5) +- PhotonShuffleExchangeSource (4) +- PhotonShuffleMapStage (3) +- PhotonShuffleExchangeSink (2) +- PhotonScan parquet workspace.default.events_source (1) (1) PhotonScan parquet workspace.default.events_source Output [4]: [id#15524L, country#15525, amount#15526, event_ts#15527] Location: PreparedDeltaFileIndex [s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/517eeec8-77b6-4174-ab67-4a65a7647c7b] ReadSchema: struct (2) PhotonShuffleExchangeSink Input [4]: [id#15524L, country#15525, amount#15526, event_ts#15527] Arguments: hashpartitioning(country#15525, 1024) (3) PhotonShuffleMapStage Input [4]: [id#15524L, country#15525, amount#15526, event_ts#15527] Arguments: REBALANCE_PARTITIONS_BY_COL, [id=#10123] (4) PhotonShuffleExchangeSource Input [4]: [id#15524L, country#15525, amount#15526, event_ts#15527] (5) PhotonColumnarToRow Input [4]: [id#15524L, country#15525, amount#15526, event_ts#15527] (6) PhotonResultStage Input [4]: [id#15524L, country#15525, amount#15526, event_ts#15527] (7) AdaptiveSparkPlan Output [4]: [id#15524L, country#15525, amount#15526, event_ts#15527] Arguments: isFinalPlan=false == Photon Explanation == The query is fully supported by Photon. == Optimizer Statistics (table names per statistics state) == missing = partial = full = events_source"


In [0]:
%sql
CREATE OR REPLACE TABLE events_source_rebalanced_country AS
SELECT /*+ REBALANCE(country) */ * FROM events_source;

num_affected_rows,num_inserted_rows


In [0]:
display_files_distribution_for_table('events_source_rebalanced_country')

file,rows
s3://dbstorage-prod-3jfyf/uc/ac35c7ae-5ee7-406f-b03d-e2b7f32ef2d0/8364d6fd-cd08-40fa-9559-33208818c727/__unitystorage/catalogs/ce27653c-8c98-43a9-a76f-6f3bc9b99acd/tables/f53097f6-49cb-437c-8c98-b0a6cfd0570c/part-00000-7762c9ac-38e6-441c-86c2-18276ff53b29.c000.zstd.parquet,1000
